In [1]:
from pymongo import MongoClient
import pandas as pd, json

In [2]:
# csv paths
users_path = 'data/PNRao_SocialMedia_Users.csv'
userprofiles_path = 'data/PNRao_SocialMedia_User_Profiles.csv'
followers_path = 'data/PNRao_SocialMedia_Followers.csv'

posts_path = 'data/PNRao_SocialMedia_Posts.csv'
shares_path = 'data/PNRao_SocialMedia_Post_Shares.csv'
likes_path = 'data/PNRao_SocialMedia_Post_Likes.csv'
comments_path = 'data/PNRao_SocialMedia_Post_Comments.csv'
hashtags_path = 'data/PNRao_SocialMedia_Post_Hashtags.csv'

hashtag_master_path = 'data/PNRao_SocialMedia_Hashtag_Master.csv'

# load dataframes
df_users = pd.read_csv(users_path)
df_profiles = pd.read_csv(userprofiles_path)
df_followers = pd.read_csv(followers_path)

df_posts = pd.read_csv(posts_path)
df_shares = pd.read_csv(shares_path)
df_likes = pd.read_csv(likes_path)
df_comments = pd.read_csv(comments_path)
df_hashtags = pd.read_csv(hashtags_path)

df_hashtag_master = pd.read_csv(hashtag_master_path)

print("dataframes successfully loaded")

dataframes successfully loaded


In [3]:
# Connect to the MongoDB server (localhost by default)
mg_client1 = MongoClient('mongodb://localhost:27017/', username='root', password='rootpwd')

##provide feedback to the user
if mg_client1:
    print("connected to mongo database successully.")
else:
    print("failed to connect to mongo database.")

connected to mongo database successully.


In [18]:
df_hashtag_master

,HashtagID,HashtagName
0,HASH-101,#travel
1,HASH-102,#foodie
2,HASH-103,#tech
3,HASH-104,#fashion
4,HASH-105,#fitness
5,HASH-106,#motivation
6,HASH-107,#crypto
7,HASH-108,#india
8,HASH-109,#photography
9,HASH-110,#art


PostID
50001    [{'HashtagID': 'HASH-106'}, {'HashtagID': 'HAS...
50002                          [{'HashtagID': 'HASH-107'}]
50003    [{'HashtagID': 'HASH-105'}, {'HashtagID': 'HAS...
50004    [{'HashtagID': 'HASH-102'}, {'HashtagID': 'HAS...
50005    [{'HashtagID': 'HASH-110'}, {'HashtagID': 'HAS...
                               ...                        
52626    [{'HashtagID': 'HASH-102'}, {'HashtagID': 'HAS...
52627                          [{'HashtagID': 'HASH-112'}]
52628    [{'HashtagID': 'HASH-101'}, {'HashtagID': 'HAS...
52629    [{'HashtagID': 'HASH-101'}, {'HashtagID': 'HAS...
52630    [{'HashtagID': 'HASH-113'}, {'HashtagID': 'HAS...
Length: 2630, dtype: object

In [22]:
# # group followers by user id
# followers_by_user = (
#     df_followers
#     .groupby("UserID")
#     .apply(lambda g: g.loc[:, ["FollowerID", "FollowerUserID"]].to_dict(orient="records"))
# )

# group comments by post_id
comments_by_post = (
    df_comments
    .groupby("PostID")
    .apply(lambda g: g.loc[:, ["CommentID", "UserID", "CommentText", "Timestamp"]].to_dict(orient="records"))
)

# group tags

tags_by_post = (
    df_hashtags
    .groupby("PostID")
    .apply(lambda g: g.loc[:, ["HashtagID"]].to_dict(orient="records"))
)

# group engagement metrics

likes_by_post = (
    df_likes
    .groupby("PostID")
    .apply(lambda g: g.loc[:, ["LikeID", "UserID","Timestamp"]].to_dict(orient="records"))
)

shares_by_post = (
    df_shares
    .groupby("PostID")
    .apply(lambda g: g.loc[:, ["ShareID", "UserID", "Timestamp"]].to_dict(orient="records"))
)

C:\Users\kirbt\AppData\Local\Temp\ipykernel_1856\1783247742.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.loc[:, ["CommentID", "UserID", "CommentText", "Timestamp"]].to_dict(orient="records"))
C:\Users\kirbt\AppData\Local\Temp\ipykernel_1856\1783247742.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.loc[:, ["HashtagID"]].to_dict(orient="records"))
C:\Users\kirbt\AppDa

In [20]:
df_tags = tags_by_post.reset_index().rename(columns={0: "Hashtags"})


In [27]:
'''
POSTS COLLECTION

Will combine data from the posts and comments csv files and insert into a singular posts collection, where comments are nested within the related post.

ATTRIBUTES:

'''

# Build final post documents
post_documents = []

for index, row in df_posts.iterrows():
    post_id = row["PostID"]

    post_doc = {
        "PostID": post_id,
        "UserID": row["UserID"],
        "PostType": row["PostType"],
        "Content": row["Content"],
        "Timestamp": row["Timestamp"],

        # embedded arrays
        "Comments": comments_by_post.get(post_id, []),
        "Hashtags": tags_by_post.get(post_id, []),

        # engagement metrics
        "LikeCount": len(likes_by_post.get(post_id, [])),
        "ShareCount": len(shares_by_post.get(post_id, [])),
    }

    post_documents.append(post_doc)

post_documents


[{'PostID': 50001,
  'UserID': 1001,
  'PostType': 'Text',
  'Content': 'Feeling motivated today to achieve my goals.',
  'Timestamp': '2024-10-12 06:16:44',
  'Comments': [],
  'Hashtags': [{'HashtagID': 'HASH-106'},
   {'HashtagID': 'HASH-109'},
   {'HashtagID': 'HASH-103'},
   {'HashtagID': 'HASH-107'}],
  'LikeCount': 16,
  'ShareCount': 0},
 {'PostID': 50002,
  'UserID': 1001,
  'PostType': 'Image',
  'Content': 'My new puppy!',
  'Timestamp': '2024-08-30 01:35:44',
  'Comments': [],
  'Hashtags': [{'HashtagID': 'HASH-107'}],
  'LikeCount': 2,
  'ShareCount': 0},
 {'PostID': 50003,
  'UserID': 1001,
  'PostType': 'Story',
  'Content': 'Behind the scenes at our office.',
  'Timestamp': '2024-07-31 19:00:42',
  'Comments': [{'CommentID': 'COM-1',
    'UserID': 1075,
    'CommentText': 'Love this!',
    'Timestamp': '2024-08-03 02:00:42'},
   {'CommentID': 'COM-2',
    'UserID': 1039,
    'CommentText': 'So true.',
    'Timestamp': '2024-08-03 02:00:42'}],
  'Hashtags': [{'HashtagID'

In [26]:
'''
USERS COLLECTION

Will combine data from the user and user profiles csv files and insert into a singular user's

ATTRIBUTES:

_id:
Username:
Name:
Bio:
Location:
IsVerified:
JoinDate:
'''

user_documents = []

for index, row in df_users.iterrows():
  user_id = row["UserID"]

  # fetch related user profile data
  profile = df_profiles[df_profiles['UserID'] == user_id].to_dict(orient='records')
  if profile:
      profile = profile[0]
  else:
      profile = {}

  user_doc = {
    "UserID": user_id,
    "Username": row["Username"],
    "Name": profile["FullName"],
    "Bio": profile["Bio"],
    "Location": profile["Location"],
    "IsVerified": profile["IsVerified"],
    "JoinDate": row["JoinDate"]
  }

  user_documents.append(user_doc)

user_documents

[{'UserID': 1001,
  'Username': 'saanvi_verma16',
  'Name': 'Saanvi Verma',
  'Bio': 'Lover of food. Based in Bengaluru.',
  'Location': 'Hyderabad',
  'IsVerified': False,
  'JoinDate': '2025-10-04'},
 {'UserID': 1002,
  'Username': 'aditya_sharma54',
  'Name': 'Aditya Sharma',
  'Bio': 'Lover of tech. Based in Kolkata.',
  'Location': 'Chennai',
  'IsVerified': True,
  'JoinDate': '2025-04-03'},
 {'UserID': 1003,
  'Username': 'reyansh_verma68',
  'Name': 'Reyansh Verma',
  'Bio': 'Lover of food. Based in Pune.',
  'Location': 'Mumbai',
  'IsVerified': False,
  'JoinDate': '2022-04-16'},
 {'UserID': 1004,
  'Username': 'aarav_mehta57',
  'Name': 'Aarav Mehta',
  'Bio': 'Lover of tech. Based in Mumbai.',
  'Location': 'Bengaluru',
  'IsVerified': False,
  'JoinDate': '2023-05-25'},
 {'UserID': 1005,
  'Username': 'reyansh_reddy30',
  'Name': 'Reyansh Reddy',
  'Bio': 'Lover of travel. Based in Pune.',
  'Location': 'Bengaluru',
  'IsVerified': False,
  'JoinDate': '2023-12-08'},
 {'Us

In [28]:
follower_documents = df_followers.to_dict(orient="records")
follower_documents

[{'FollowerID': 1, 'UserID': 1037, 'FollowerUserID': 1001},
 {'FollowerID': 2, 'UserID': 1028, 'FollowerUserID': 1001},
 {'FollowerID': 3, 'UserID': 1031, 'FollowerUserID': 1001},
 {'FollowerID': 4, 'UserID': 1053, 'FollowerUserID': 1001},
 {'FollowerID': 5, 'UserID': 1022, 'FollowerUserID': 1001},
 {'FollowerID': 6, 'UserID': 1041, 'FollowerUserID': 1001},
 {'FollowerID': 7, 'UserID': 1021, 'FollowerUserID': 1001},
 {'FollowerID': 8, 'UserID': 1050, 'FollowerUserID': 1001},
 {'FollowerID': 9, 'UserID': 1057, 'FollowerUserID': 1001},
 {'FollowerID': 10, 'UserID': 1071, 'FollowerUserID': 1001},
 {'FollowerID': 11, 'UserID': 1042, 'FollowerUserID': 1001},
 {'FollowerID': 12, 'UserID': 1086, 'FollowerUserID': 1001},
 {'FollowerID': 13, 'UserID': 1050, 'FollowerUserID': 1001},
 {'FollowerID': 14, 'UserID': 1049, 'FollowerUserID': 1001},
 {'FollowerID': 15, 'UserID': 1077, 'FollowerUserID': 1001},
 {'FollowerID': 16, 'UserID': 1016, 'FollowerUserID': 1001},
 {'FollowerID': 17, 'UserID': 107

In [30]:
db = mg_client1["socialmedia"]

posts_collection = db["posts"]
users_collection = db["users"]
likes_collection = db["likes"]
shares_collection = db["shares"]
followers_collection = db["followers"]
hashtags_collection = db["hashtags"]

posts_collection.insert_many(post_documents)
users_collection.insert_many(user_documents)
followers_collection.insert_many(follower_documents)
likes_collection.insert_many(df_likes.to_dict(orient="records"))
shares_collection.insert_many(df_shares.to_dict(orient="records"))
hashtags_collection.insert_many(df_hashtag_master.to_dict(orient="records"))



InsertManyResult([ObjectId('69b23e182da4ddcf6501fa51'), ObjectId('69b23e182da4ddcf6501fa52'), ObjectId('69b23e182da4ddcf6501fa53'), ObjectId('69b23e182da4ddcf6501fa54'), ObjectId('69b23e182da4ddcf6501fa55'), ObjectId('69b23e182da4ddcf6501fa56'), ObjectId('69b23e182da4ddcf6501fa57'), ObjectId('69b23e182da4ddcf6501fa58'), ObjectId('69b23e182da4ddcf6501fa59'), ObjectId('69b23e182da4ddcf6501fa5a'), ObjectId('69b23e182da4ddcf6501fa5b'), ObjectId('69b23e182da4ddcf6501fa5c'), ObjectId('69b23e182da4ddcf6501fa5d')], acknowledged=True)